# Welcome to Writing SQL 101

This a 1 hour tutorial to understand the basic of SQL.

## Is SQL a standard?

SQL is a languaje with different dialect: every engine implement some "version" of SQL but usually add some customization to it.

**More info:** https://en.wikipedia.org/wiki/SQL#Standardization_history

## Where can I use SQL?

- Relational databases:
    - Oracle
    - Microsoft SQL Server
    - PostGres
- Data warehouse
    - AWS Redshift
    - Google BigQuery
    - Apache Doris
- Self-container
    - SQLite
- Distributed engines
    - Google Cloud Spanner
    - CockroachDB
    - Apache Spark

## Let's start!

Let's create and Apache SparkSession

In [ ]:
from pyspark.sql import SparkSession
SPARK_CONFIG = {
    "spark.sql.repl.eagerEval.enabled": True,
    "spark.sql.repl.eagerEval.maxNumRows": 20,
    "spark.sql.repl.eagerEval.truncate": 50,
}

builder = SparkSession.builder
for key, value in SPARK_CONFIG.items():
    builder = builder.config(key, value)
spark = builder.getOrCreate()
spark.version

## Create test data


### Users

We get the data form a small MiniMarket. It a small bussiness with only 3 working people.

In [ ]:
users = spark.createDataFrame([
    {"id": 1, "name": "Carlos", "admin_role": True},
    {"id": 2, "name": "Alejandra", "admin_role": False},
    {"id": 3, "name": "Sara", "admin_role": False},
    {"id": 4, "name": "Romi", "admin_role": False},
])
users.createTempView("users")
users

### Sales

Different days are stored separated.

In [ ]:
from datetime import datetime
sales1 = spark.createDataFrame([
    {"id": 1, "time": datetime(2026, 6, 27, 8, 0, 0), "user_id": 1, "amount": 100, "product": "Marraqueta"},
    {"id": 2, "time": datetime(2026, 6, 27, 10, 0, 0), "user_id": 1, "amount": 200, "product": "Pan de molde"},
    {"id": 3, "time": datetime(2026, 6, 27, 12, 0, 0), "user_id": 2, "amount": 300, "product": "Medias Lunas"},
])
sales1.createTempView("sales1")
sales1

In [ ]:
sales2 = spark.createDataFrame([
    {"id": 4, "time": datetime(2026, 6, 30, 8, 0, 0), "user_id": 3, "amount": 100, "product": "Marraqueta"},
    {"id": 5, "time": datetime(2026, 6, 30, 10, 0, 0), "user_id": 3, "amount": 200, "product": "Pan de molde"},
    {"id": 6, "time": datetime(2026, 6, 30, 12, 0, 0), "user_id": 2, "amount": 300, "product": "Medias Luneas"},
    {"id": 7, "time": datetime(2026, 6, 30, 12, 10, 0), "user_id": 2, "amount": 300, "product": "Cafe"},
    {"id": 8, "time": datetime(2026, 6, 30, 13, 0, 0), "user_id": 3, "amount": 10, "product": "Fosforos"},
    {"id": 9, "time": datetime(2026, 6, 30, 14, 0, 0), "user_id": 3, "amount": 10, "product": "Caramelos"},
    {"id": 10, "time": datetime(2026, 6, 30, 18, 0, 0), "user_id": 3, "amount": 200, "product": "Agua"},
])
sales2.createTempView("sales2")
sales2

### Cancelled sales

Sales are "soft deleted"

In [ ]:
cancelled_sales = spark.createDataFrame([
    {"id": 2},
    {"id": 10},
])
cancelled_sales.createTempView("cancelled_sales")
cancelled_sales

## Let's write our first SQL!

SQL ignore line jumps and white spaces / tabs.

We format it for easier read.

```
SELECT
[columns list or * for all]
FROM
[table name]
```

In [ ]:
SQL = """
SELECT
    id,
    time,
    product,
    amount,
    user_id
FROM sales1
"""
spark.sql(SQL)

### Functions

Let's calculate the date using time column

In [ ]:
SQL = """
SELECT
    id,
    cast(time as date) as date,
    time,
    product,
    amount,
    user_id
FROM sales1
"""
spark.sql(SQL)

### Where

The conditions to meet. Is evaluated per row.

Mutiples conditions can be chained using `AND` and `OR`.

Negation with `NOT`

Multiples options using `IS IN`
Text patterns using `LIKE` (wildcard `%`)

In [ ]:
SQL = """
SELECT
    id, time, product, amount, user_id
FROM sales1
WHERE user_id = 1 AND amount > 0
"""
spark.sql(SQL)

## Order

DESC = descending

ASC or ommit = ascending

In [ ]:
SQL = """
SELECT
    id, time, product, amount, user_id
FROM sales1
ORDER By amount DESC
"""
spark.sql(SQL)

# Group By

In [ ]:
SQL = """
SELECT
    admin_role,
    count(id) as total_users
FROM users
GROUP by admin_role
"""
spark.sql(SQL)

## Having

Not common to use

In [ ]:
SQL = """
SELECT
    admin_role,
    count(id) as total_users
FROM users
GROUP by admin_role
HAVING total_users > 1
"""
spark.sql(SQL)

## Union

Concatenate rows from multiples tables

In [ ]:
SQL = """
SELECT
    id, time, product, amount, user_id
FROM sales1

UNION

SELECT
    id, time, product, amount, user_id
FROM sales2
"""
spark.sql(SQL)

## With

Work as temporary table

In [ ]:
SQL = """
WITH sales as (
    SELECT
        id, time, product, amount, user_id
    FROM sales1

    UNION

    SELECT
        id, time, product, amount, user_id
    FROM sales2
)
SELECT
    id,
    cast(time as date) as date,
    time,
    product,
    amount,
    user_id
FROM sales
"""
spark.sql(SQL)

## Subquery

In [ ]:
SQL = """
SELECT
    id,
    cast(time as date) as date,
    time,
    product,
    amount,
    user_id
FROM (
    SELECT
        id, time, product, amount, user_id
    FROM sales1

    UNION

    SELECT
        id, time, product, amount, user_id
    FROM sales2
)
"""
spark.sql(SQL)

In [ ]:
SQL = """
WITH sales as (
    SELECT
        id, time, product, amount, user_id
    FROM sales1

    UNION

    SELECT
        id, time, product, amount, user_id
    FROM sales2
)
SELECT
    id,
    cast(time as date) as date,
    time,
    product,
    amount,
    user_id
FROM sales
WHERE id not in (SELECT id FROM cancelled_sales)
"""
spark.sql(SQL)

## Joins

- inner
- full
- left
- right
- anti

In [ ]:
SQL = """
WITH sales as (
    SELECT
        id, time, product, amount, user_id
    FROM sales1

    UNION

    SELECT
        id, time, product, amount, user_id
    FROM sales2
)
SELECT
    sales.id,
    cast(sales.time as date) as date,
    sales.time,
    sales.product,
    sales.amount,
    users.name
FROM sales
RIGHT JOIN users
ON sales.user_id = users.id
"""
spark.sql(SQL)

## Window functions

In [ ]:
SQL = """
WITH sales as (
    SELECT
        id, time, product, amount, user_id
    FROM sales1

    UNION

    SELECT
        id, time, product, amount, user_id
    FROM sales2
),
sales_date as (
    SELECT
        sales.id,
        cast(sales.time as date) as date,
        sales.time,
        sales.product,
        sales.amount,
        users.name
    FROM sales
    LEFT JOIN users
    ON sales.user_id = users.id
)
SELECT
    *,
    rank() OVER (PARTITION BY date ORDER BY amount DESC) as rank
FROM sales_date
ORDER BY rank
"""
spark.sql(SQL)